In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

from pptx import Presentation
from pptx.util import Inches

# =========================================================
# CONFIG
# =========================================================

OUTPUT_DIR = "output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

EXCEL_FILE = os.path.join(
    OUTPUT_DIR,
    "Retail_Executive_Report.xlsx"
)

PPT_FILE = os.path.join(
    OUTPUT_DIR,
    "Retail_Executive_Report.pptx"
)

# =========================================================
# GENERATE SAMPLE DATA
# =========================================================

np.random.seed(42)

rows = 5000

regions = ["North", "South", "East", "West"]

categories = [
    "Technology",
    "Furniture",
    "Office Supplies"
]

products = [
    "Laptop",
    "Monitor",
    "Chair",
    "Desk",
    "Printer",
    "Phone",
    "Tablet",
    "Keyboard"
]

df = pd.DataFrame({
    "OrderDate": np.random.choice(
        pd.date_range(
            "2024-01-01",
            "2024-12-31",
            freq="D"
        ),
        rows
    ),
    "Region": np.random.choice(
        regions,
        rows
    ),
    "Category": np.random.choice(
        categories,
        rows
    ),
    "Product": np.random.choice(
        products,
        rows
    ),
    "Quantity": np.random.randint(
        1,
        10,
        rows
    ),
    "UnitPrice": np.random.randint(
        50,
        1500,
        rows
    )
})

df["Sales"] = (
    df["Quantity"] *
    df["UnitPrice"]
)

df["Profit"] = (
    df["Sales"] *
    np.random.uniform(
        0.10,
        0.35,
        rows
    )
)

df["Month"] = (
    pd.to_datetime(df["OrderDate"])
    .dt.to_period("M")
    .astype(str)
)

# =========================================================
# KPI
# =========================================================

total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = len(df)
avg_order_value = total_sales / total_orders

# =========================================================
# SUMMARIES
# =========================================================

sales_region = (
    df.groupby("Region")["Sales"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

profit_region = (
    df.groupby("Region")["Profit"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

sales_category = (
    df.groupby("Category")["Sales"]
      .sum()
      .reset_index()
)

monthly_sales = (
    df.groupby("Month")["Sales"]
      .sum()
      .reset_index()
)

top_products = (
    df.groupby("Product")["Sales"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
      .reset_index()
)

# =========================================================
# WRITE EXCEL
# =========================================================

with pd.ExcelWriter(
        EXCEL_FILE,
        engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        sheet_name="Raw_Data",
        index=False
    )

    sales_region.to_excel(
        writer,
        sheet_name="Sales_By_Region",
        index=False
    )

    profit_region.to_excel(
        writer,
        sheet_name="Profit_By_Region",
        index=False
    )

    sales_category.to_excel(
        writer,
        sheet_name="Sales_By_Category",
        index=False
    )

    monthly_sales.to_excel(
        writer,
        sheet_name="Monthly_Sales",
        index=False
    )

    top_products.to_excel(
        writer,
        sheet_name="Top_Products",
        index=False
    )

print("Excel workbook created")

# =========================================================
# FORMAT EXCEL
# =========================================================

wb = load_workbook(EXCEL_FILE)

header_fill = PatternFill(
    fill_type="solid",
    start_color="1F4E78"
)

header_font = Font(
    color="FFFFFF",
    bold=True
)

for ws in wb.worksheets:

    for cell in ws[1]:

        cell.fill = header_fill
        cell.font = header_font

    for column in ws.columns:

        max_len = 0

        for cell in column:

            try:
                max_len = max(
                    max_len,
                    len(str(cell.value))
                )
            except:
                pass

        ws.column_dimensions[
            column[0].column_letter
        ].width = max_len + 3

wb.save(EXCEL_FILE)

print("Excel formatting applied")

# =========================================================
# CHARTS
# =========================================================

charts = []

def save_chart(name):
    path = os.path.join(
        OUTPUT_DIR,
        f"{name}.png"
    )
    plt.tight_layout()
    plt.savefig(path)
    plt.close()
    charts.append(path)

# 1

plt.figure(figsize=(8,5))
plt.bar(
    sales_region["Region"],
    sales_region["Sales"]
)
plt.title("Sales by Region")
save_chart("sales_region")

# 2

plt.figure(figsize=(8,5))
plt.bar(
    profit_region["Region"],
    profit_region["Profit"]
)
plt.title("Profit by Region")
save_chart("profit_region")

# 3

plt.figure(figsize=(8,5))
plt.bar(
    sales_category["Category"],
    sales_category["Sales"]
)
plt.title("Sales by Category")
save_chart("sales_category")

# 4

plt.figure(figsize=(10,5))
plt.plot(
    monthly_sales["Month"],
    monthly_sales["Sales"],
    marker="o"
)

plt.xticks(rotation=45)

plt.title("Monthly Sales Trend")

save_chart("monthly_sales")

# 5

plt.figure(figsize=(8,5))
plt.bar(
    top_products["Product"],
    top_products["Sales"]
)

plt.title("Top Products")

save_chart("top_products")

print("Charts generated")

# =========================================================
# POWERPOINT
# =========================================================

prs = Presentation()

# ---------------------------------------------------------
# COVER
# ---------------------------------------------------------

slide = prs.slides.add_slide(
    prs.slide_layouts[0]
)

slide.shapes.title.text = (
    "Retail Executive Dashboard"
)

slide.placeholders[1].text = (
    "Generated Automatically with Python"
)

# ---------------------------------------------------------
# EXEC SUMMARY
# ---------------------------------------------------------

slide = prs.slides.add_slide(
    prs.slide_layouts[5]
)

slide.shapes.title.text = (
    "Executive Summary"
)

textbox = slide.shapes.add_textbox(
    Inches(0.8),
    Inches(1.3),
    Inches(5),
    Inches(2)
)

textbox.text_frame.text = f"""
Total Sales: ${total_sales:,.0f}

Total Profit: ${total_profit:,.0f}

Total Orders: {total_orders:,}

Average Order Value:
${avg_order_value:,.2f}
"""

# ---------------------------------------------------------
# CHART SLIDES
# ---------------------------------------------------------

titles = [
    "Sales by Region",
    "Profit by Region",
    "Sales by Category",
    "Monthly Sales Trend",
    "Top Products"
]

for title, chart in zip(
        titles,
        charts
):

    slide = prs.slides.add_slide(
        prs.slide_layouts[5]
    )

    slide.shapes.title.text = title

    slide.shapes.add_picture(
        chart,
        Inches(0.5),
        Inches(1),
        width=Inches(8.5)
    )

# ---------------------------------------------------------
# INSIGHTS
# ---------------------------------------------------------

slide = prs.slides.add_slide(
    prs.slide_layouts[5]
)

slide.shapes.title.text = (
    "Key Business Insights"
)

textbox = slide.shapes.add_textbox(
    Inches(0.7),
    Inches(1.3),
    Inches(8),
    Inches(3)
)

textbox.text_frame.text = f"""
Highest Revenue Region:
{sales_region.iloc[0]['Region']}

Highest Profit Region:
{profit_region.iloc[0]['Region']}

Top Selling Product:
{top_products.iloc[0]['Product']}

Revenue Generated:
${sales_region.iloc[0]['Sales']:,.0f}
"""

# ---------------------------------------------------------
# THANK YOU
# ---------------------------------------------------------

slide = prs.slides.add_slide(
    prs.slide_layouts[5]
)

slide.shapes.title.text = "Thank You"

textbox = slide.shapes.add_textbox(
    Inches(1),
    Inches(2),
    Inches(6),
    Inches(1)
)

textbox.text_frame.text = (
    "End of Automated Executive Report"
)

# =========================================================
# SAVE PPT
# =========================================================

prs.save(PPT_FILE)

print("\n====================================")
print("REPORT GENERATED SUCCESSFULLY")
print("====================================")
print("Excel:", EXCEL_FILE)
print("PowerPoint:", PPT_FILE)
print("====================================")

Excel workbook created
Excel formatting applied
Charts generated

REPORT GENERATED SUCCESSFULLY
Excel: output/Retail_Executive_Report.xlsx
PowerPoint: output/Retail_Executive_Report.pptx


In [3]:
print(os.getcwd())

/Users/kanchana/Desktop/VS Code/Pandas
